In [ ]:
# Local Greedy Search (Hill Climbing) for N-Queens Problem
# Measures execution time, memory usage, iterations, success

import random
import time
import tracemalloc

# Count conflicts in current board
def count_conflicts(board):
    n = len(board)
    conflicts = 0

    for i in range(n):
        for j in range(i + 1, n):
            if board[i] == board[j]:  # same column
                conflicts += 1
            elif abs(board[i] - board[j]) == abs(i - j):  # diagonal
                conflicts += 1

    return conflicts

# Generate best neighbor by moving one queen
def best_neighbor(board):
    n = len(board)
    current_conflicts = count_conflicts(board)

    best_board = board[:]
    best_score = current_conflicts

    for row in range(n):
        original_col = board[row]

        for col in range(n):
            if col == original_col:
                continue

            board[row] = col
            score = count_conflicts(board)

            if score < best_score:
                best_score = score
                best_board = board[:]

        board[row] = original_col

    return best_board, best_score

# Hill Climbing Algorithm
def greedy_nqueens(n, max_restarts=50):
    iterations = 0

    for restart in range(max_restarts):
        board = [random.randint(0, n - 1) for _ in range(n)]

        while True:
            iterations += 1
            current_score = count_conflicts(board)

            if current_score == 0:
                return board, True, iterations

            neighbor, neighbor_score = best_neighbor(board)

            # If no improvement, restart
            if neighbor_score >= current_score:
                break

            board = neighbor

    return None, False, iterations

# Print Board
def print_board(solution):
    n = len(solution)
    for row in range(n):
        line = ""
        for col in range(n):
            if solution[row] == col:
                line += "Q "
            else:
                line += ". "
        print(line)
    print()

# Run Test
def run_test(n):
    print("=" * 60)
    print(f"Running Greedy Search for N = {n}")

    tracemalloc.start()
    start_time = time.time()

    solution, success, iterations = greedy_nqueens(n)

    end_time = time.time()
    current, peak = tracemalloc.get_traced_memory()
    tracemalloc.stop()

    execution_time = end_time - start_time
    memory_used = peak / (1024 * 1024)

    print(f"Success         : {success}")
    print(f"Iterations      : {iterations}")
    print(f"Execution Time  : {execution_time:.4f} sec")
    print(f"Memory Used     : {memory_used:.4f} MB")

    if success and n <= 30:
        print("\nSolution:")
        print_board(solution)

# Main
if __name__ == "__main__":

    test_cases = [10, 30, 50, 100, 200, 500]

    for n in test_cases:
        run_test(n)

In [ ]:
# dfs_nqueens.py
# Exhaustive Search using Depth First Search (Backtracking)
# Solving N-Queens Problem
# Measures execution time, memory usage, iterations, success

import time
import tracemalloc

# DFS Solver Class
class NQueensDFS:
    def __init__(self, n):
        self.n = n
        self.board = [-1] * n
        self.solutions = []
        self.iterations = 0

    # Check if queen placement is safe
    def is_safe(self, row, col):
        for prev_row in range(row):
            prev_col = self.board[prev_row]

            # Same column
            if prev_col == col:
                return False

            # Diagonal attack
            if abs(prev_col - col) == abs(prev_row - row):
                return False

        return True

    # DFS Backtracking
    def solve(self, row=0):
        self.iterations += 1

        if row == self.n:
            self.solutions.append(self.board[:])
            return True

        for col in range(self.n):
            if self.is_safe(row, col):
                self.board[row] = col

                if self.solve(row + 1):
                    return True   # Stop after first solution

                self.board[row] = -1

        return False

# Display Board
def print_board(solution):
    n = len(solution)
    for row in range(n):
        line = ""
        for col in range(n):
            if solution[row] == col:
                line += "Q "
            else:
                line += ". "
        print(line)
    print()

# Run Test
def run_test(n):
    print("=" * 60)
    print(f"Running DFS for N = {n}")

    solver = NQueensDFS(n)

    tracemalloc.start()
    start_time = time.time()

    success = solver.solve()

    end_time = time.time()
    current, peak = tracemalloc.get_traced_memory()
    tracemalloc.stop()

    execution_time = end_time - start_time
    memory_used = peak / (1024 * 1024)  # MB

    print(f"Success         : {success}")
    print(f"Iterations      : {solver.iterations}")
    print(f"Execution Time  : {execution_time:.4f} sec")
    print(f"Memory Used     : {memory_used:.4f} MB")

    if success:
        print("\nSolution:")
        print_board(solver.solutions[0])

# Main
if __name__ == "__main__":

    test_cases = [10, 30, 50, 100, 200, 500]

    for n in test_cases:
        # DFS becomes too slow for very large N
        if n > 30:
            print("=" * 60)
            print(f"Skipping DFS for N = {n} (Too Expensive Computationally)")
            continue

        run_test(n)

In [ ]:
# simulated_annealing_nqueens.py
# Simulated Annealing for N-Queens Problem
# Measures execution time, memory usage, iterations, success

import random
import math
import time
import tracemalloc

# Count conflicts in board
# board[row] = column position of queen
def count_conflicts(board):
    n = len(board)
    conflicts = 0

    for i in range(n):
        for j in range(i + 1, n):
            if board[i] == board[j]:
                conflicts += 1
            elif abs(board[i] - board[j]) == abs(i - j):
                conflicts += 1

    return conflicts

# Generate random neighbor
def random_neighbor(board):
    n = len(board)
    new_board = board[:]

    row = random.randint(0, n - 1)
    col = random.randint(0, n - 1)

    new_board[row] = col
    return new_board

# Simulated Annealing Solver
def simulated_annealing(n, temperature=100.0, cooling_rate=0.995, min_temp=0.001, max_steps=200000):
    board = [random.randint(0, n - 1) for _ in range(n)]

    current_conflicts = count_conflicts(board)
    iterations = 0

    while temperature > min_temp and iterations < max_steps:

        if current_conflicts == 0:
            return board, True, iterations

        neighbor = random_neighbor(board)
        neighbor_conflicts = count_conflicts(neighbor)

        delta = neighbor_conflicts - current_conflicts

        # Accept better solution
        if delta < 0:
            board = neighbor
            current_conflicts = neighbor_conflicts

        else:
            probability = math.exp(-delta / temperature)

            if random.random() < probability:
                board = neighbor
                current_conflicts = neighbor_conflicts

        temperature *= cooling_rate
        iterations += 1

    if current_conflicts == 0:
        return board, True, iterations

    return board, False, iterations

# Print Board
def print_board(solution):
    n = len(solution)
    for row in range(n):
        line = ""
        for col in range(n):
            if solution[row] == col:
                line += "Q "
            else:
                line += ". "
        print(line)
    print()

# Run Test
def run_test(n):
    print("=" * 60)
    print(f"Running Simulated Annealing for N = {n}")

    tracemalloc.start()
    start_time = time.time()

    solution, success, iterations = simulated_annealing(n)

    end_time = time.time()
    current, peak = tracemalloc.get_traced_memory()
    tracemalloc.stop()

    execution_time = end_time - start_time
    memory_used = peak / (1024 * 1024)

    print(f"Success         : {success}")
    print(f"Iterations      : {iterations}")
    print(f"Execution Time  : {execution_time:.4f} sec")
    print(f"Memory Used     : {memory_used:.4f} MB")

    if success and n <= 30:
        print("\nSolution:")
        print_board(solution)

# Main
if __name__ == "__main__":

    test_cases = [10, 30, 50, 100, 200, 500]

    for n in test_cases:
        run_test(n)

In [ ]:
# genetic_nqueens.py
# Genetic Algorithm for N-Queens Problem
# Measures execution time, memory usage, iterations, success

import random
import time
import tracemalloc

# Fitness Function
# Higher fitness = fewer conflicts
def fitness(board):
    n = len(board)
    conflicts = 0

    for i in range(n):
        for j in range(i + 1, n):
            if board[i] == board[j]:
                conflicts += 1
            elif abs(board[i] - board[j]) == abs(i - j):
                conflicts += 1

    max_pairs = n * (n - 1) // 2
    return max_pairs - conflicts

# Create random chromosome
def create_individual(n):
    return [random.randint(0, n - 1) for _ in range(n)]

# Selection (Tournament)
def selection(population):
    tournament = random.sample(population, 3)
    tournament.sort(key=lambda x: fitness(x), reverse=True)
    return tournament[0]

# Crossover
def crossover(parent1, parent2):
    n = len(parent1)
    point = random.randint(1, n - 2)

    child1 = parent1[:point] + parent2[point:]
    child2 = parent2[:point] + parent1[point:]

    return child1, child2

# Mutation
def mutate(board, mutation_rate=0.05):
    n = len(board)

    for i in range(n):
        if random.random() < mutation_rate:
            board[i] = random.randint(0, n - 1)

# Genetic Algorithm Solver
def genetic_algorithm(n, population_size=100, generations=1000):
    population = [create_individual(n) for _ in range(population_size)]
    max_fitness = n * (n - 1) // 2

    for generation in range(generations):

        population.sort(key=lambda x: fitness(x), reverse=True)

        # Found solution
        if fitness(population[0]) == max_fitness:
            return population[0], True, generation + 1

        new_population = population[:10]   # elitism

        while len(new_population) < population_size:
            parent1 = selection(population)
            parent2 = selection(population)

            child1, child2 = crossover(parent1, parent2)

            mutate(child1)
            mutate(child2)

            new_population.append(child1)

            if len(new_population) < population_size:
                new_population.append(child2)

        population = new_population

    population.sort(key=lambda x: fitness(x), reverse=True)
    return population[0], False, generations

# Print Board
def print_board(solution):
    n = len(solution)
    for row in range(n):
        line = ""
        for col in range(n):
            if solution[row] == col:
                line += "Q "
            else:
                line += ". "
        print(line)
    print()

# Run Test
def run_test(n):
    print("=" * 60)
    print(f"Running Genetic Algorithm for N = {n}")

    tracemalloc.start()
    start_time = time.time()

    solution, success, generations = genetic_algorithm(n)

    end_time = time.time()
    current, peak = tracemalloc.get_traced_memory()
    tracemalloc.stop()

    execution_time = end_time - start_time
    memory_used = peak / (1024 * 1024)

    print(f"Success         : {success}")
    print(f"Generations     : {generations}")
    print(f"Execution Time  : {execution_time:.4f} sec")
    print(f"Memory Used     : {memory_used:.4f} MB")

    if success and n <= 30:
        print("\nSolution:")
        print_board(solution)

# Main
if __name__ == "__main__":

    test_cases = [10, 30, 50, 100, 200, 500]

    for n in test_cases:
        run_test(n)

Running Genetic Algorithm for N = 10
Success         : True
Generations     : 415
Execution Time  : 11.8771 sec
Memory Used     : 0.0311 MB

Solution:
Q . . . . . . . . . 
. . . . . Q . . . . 
. . . . . . . Q . . 
. Q . . . . . . . . 
. . . . . . Q . . . 
. . . . . . . . Q . 
. . Q . . . . . . . 
. . . . Q . . . . . 
. . . . . . . . . Q 
. . . Q . . . . . . 

Running Genetic Algorithm for N = 30
Success         : False
Generations     : 1000
Execution Time  : 246.0022 sec
Memory Used     : 0.0566 MB
Running Genetic Algorithm for N = 50
Success         : False
Generations     : 1000
Execution Time  : 727.6076 sec
Memory Used     : 0.0858 MB
Running Genetic Algorithm for N = 100
Success         : False
Generations     : 1000
Execution Time  : 3197.5318 sec
Memory Used     : 0.1604 MB
Running Genetic Algorithm for N = 200


# New Section